<a href="https://colab.research.google.com/github/Pmskabir1234/Torch/blob/main/Fashion_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [11]:
training_data = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=ToTensor()
)
test_data = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=ToTensor()
)

In [44]:
batch_size = 64
train_dataloader = DataLoader(training_data,batch_size=batch_size)
test_dataloader = DataLoader(test_data,batch_size=batch_size)

In [24]:
for X,y in train_dataloader:
  print(f"Shape of X: {X.shape}")
  print(f"Shape of y: {y.shape}")
  break

Shape of X: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64])


In [45]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [48]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28,512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )
  def forward(self,x):
    x = self.flatten(x)
    x = self.linear_relu_stack(x)
    return x

model = NeuralNetwork().to(device)

In [49]:
for name,param in model.named_parameters():
  print(name,param.shape)
print("Total params: ",sum(p.numel() for p in model.parameters()))

linear_relu_stack.0.weight torch.Size([512, 784])
linear_relu_stack.0.bias torch.Size([512])
linear_relu_stack.2.weight torch.Size([512, 512])
linear_relu_stack.2.bias torch.Size([512])
linear_relu_stack.4.weight torch.Size([256, 512])
linear_relu_stack.4.bias torch.Size([256])
linear_relu_stack.6.weight torch.Size([10, 256])
linear_relu_stack.6.bias torch.Size([10])
Total params:  798474


In [55]:
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

In [59]:
def train(dataloader, model, loss_func, optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch,(X,y) in enumerate(dataloader):
    X,y = X.to(device), y.to(device)


    optimizer.zero_grad()

    #prediction and error computation
    pred = model(X)
    loss = loss_func(pred, y)

    #backpropagation of adjust weights and reduce error
    loss.backward()
    optimizer.step()


    if batch%100 == 0:
      loss, current = loss.item(), (batch+1)*len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")



In [57]:
def test(dataloader, model, loss_func):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0,0
  with torch.no_grad():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_func(pred, y).item()
      correct += (pred.argmax(1)==y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Remarks: \n Accuracy: {(100*correct):>0.1f}%, Avg loss:{test_loss:>8f} \n")

In [60]:
epochs = 5
for t in range(epochs):
  print(f"Epoch {t+1} \n _________________")
  train(train_dataloader, model, loss_func, optimizer)
  test(test_dataloader, model, loss_func)

print("Training Done!")

Epoch 1 
 _________________
loss: 0.122191 [   64/60000]
loss: 0.178552 [ 6464/60000]
loss: 0.160186 [12864/60000]
loss: 0.212663 [19264/60000]
loss: 0.346251 [25664/60000]
loss: 0.204809 [32064/60000]
loss: 0.187578 [38464/60000]
loss: 0.260409 [44864/60000]
loss: 0.239444 [51264/60000]
loss: 0.286900 [57664/60000]
Remarks: 
 Accuracy: 87.2%, Avg loss:0.359406 

Epoch 2 
 _________________
loss: 0.168095 [   64/60000]
loss: 0.175667 [ 6464/60000]
loss: 0.155484 [12864/60000]
loss: 0.177373 [19264/60000]
loss: 0.354616 [25664/60000]
loss: 0.191230 [32064/60000]
loss: 0.206120 [38464/60000]
loss: 0.256891 [44864/60000]
loss: 0.229421 [51264/60000]
loss: 0.376790 [57664/60000]
Remarks: 
 Accuracy: 87.8%, Avg loss:0.357408 

Epoch 3 
 _________________
loss: 0.197763 [   64/60000]
loss: 0.184151 [ 6464/60000]
loss: 0.196797 [12864/60000]
loss: 0.169320 [19264/60000]
loss: 0.316701 [25664/60000]
loss: 0.240235 [32064/60000]
loss: 0.168454 [38464/60000]
loss: 0.246221 [44864/60000]
loss: 0.

##Notes

1. pred.argmax has shape of (64,10)
2. Experimented and noticed that Adam works almost 10 times better than SGD in accuracy in this scenerio.
3. model.train: Learning activated mode, Dropout works and BatchNorm updates.
4. model.eval: Inference mode, Dropout does not work & BatchNorm does not updates
5. loss.backward computes gradients to minimize loss
6. optimizer.step updates calculated weights
7. optimizer.zero_grad clears old gradient to improve learning


In [61]:
torch.save(model.state_dict(), "model.pth")
print("PyTorch Model saved State to model.pth")

PyTorch Model saved State to model.pth


In [64]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [71]:
classes = [
    "Tshirt/Top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    'Shirt',
    "Sneaker",
    "Bag",
    'Ankle Boot'
]
model.eval()
X,y = test_data[69][0], test_data[69][1]
with torch.no_grad():
  X = X.to(device)
  pred = model(X)
  predicted, real = classes[pred[0].argmax(0)], classes[y]
  print(f"Predicted: {predicted} & Real: {real}")

Predicted: Bag & Real: Bag
